# Notebook 02 — SMAP preprocessing

*Corrected version of the original `02_preprocessing.ipynb`. It also replaces
`02c_smap_generate_channel_data_kaggle.ipynb`, which was a Kaggle copy of the same steps:
this notebook finds its inputs on Kaggle by itself.*

**What this notebook does.** It scales every SMAP channel, cuts windows, builds per-timestep
labels for the test files, fixes the channel split, and saves everything for the later
notebooks.

**Why.** The original preprocessing produced the data behind the manuscript's SMAP table.
The corrected version keeps the same scaling and window rules, so the old numbers can be
reproduced, and adds what a fair evaluation needs: the full scaled test series with
per-timestep labels.

**Input.** The raw SMAP release (found automatically).

**Output.** `smap_prepared.pkl`, `smap_split.json` and `smap_preprocessing_config.json`.

### What was corrected

1. **The split no longer depends on cell order.** The original shuffled all 81 channels
   (SMAP and MSL) twice with Python's global random state, and the original notebook 04 then removed the
   MSL channels, leaving a 39 / 6 / 9 split. That split is kept (so results stay comparable)
   but is now written out explicitly in `maml_common.py`.
2. **Only SMAP is processed.** MSL channels have 55 features and were never usable by the
   25-feature model.
3. **P-2's two CSV rows are merged.** The original used only the first row. P-2 is a
   meta-training channel, so this does not change any evaluation result.
4. **`ast.literal_eval` replaces `eval`** for reading the anomaly ranges.
5. **The full scaled test series and per-timestep labels are saved.** The original saved
   only windows: anomaly windows from the test file and normal windows from the training
   file. Scoring normal windows from one file against anomaly windows from another mixes
   up "anomalous" with "comes from the test period" (see the audit in the README).
6. The stale `channel_data.pkl` (unscaled) and a `config.json` whose query sizes (10
   anomalies, 50 normals) were never used are no longer produced.
7. The exclusions of D-12 and P-4 are recorded with their reasons. The P-4 reason came from
   looking at test anomalies, so it is marked as post hoc and P-4 is still scored separately.

**Kept exactly as before:** MinMax scaling fitted on each channel's training file only,
clipping to [0, 1], window length 30, normal windows every 10 steps from the training
file, legacy anomaly windows every 5 steps from `test[start : end + 30]`. (The original
also capped normal windows at 500 per channel; no SMAP channel has more than 286, so the
cap never applied.)

In [1]:
import os, sys

def _find_common():
    """Find maml_common.py: this folder when run locally, /kaggle/input on Kaggle."""
    for root in [os.getcwd(), "/kaggle/input"]:
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "maml_common.py" in files:
                    return d
    raise FileNotFoundError("maml_common.py not found. Run from the Corrected-Notebooks folder, "
                            "or attach that folder to Kaggle as a Dataset.")

sys.path.insert(0, _find_common())
import maml_common as sc
SMOKE = os.environ.get("SMAP_SMOKE") == "1"     # tiny settings for testing only
OUT = sc.output_dir(smoke=SMOKE)
print("shared code:", sc.__file__)
print("outputs go to:", OUT)
print("code version:", sc.git_commit())

import pickle
import numpy as np, pandas as pd

shared code: /home/user/Objective-2/Corrected-Notebooks/maml_common.py
outputs go to: /home/user/Objective-2/Corrected-Notebooks/outputs
code version: 85c9644f9e8f5ee6eae4e65bf032fc8577223ed3


## 1 — Load and prepare all SMAP channels

In [2]:
data, labels = sc.build_smap_dataset()
print("SMAP channels prepared:", len(data))
assert len(data) == 54
assert all(d["train"].shape[1] == 25 for d in data.values())

SMAP channels prepared: 54


## 2 — The channel split

Meta-train channels are used to learn the starting point, meta-validation channels to pick
checkpoints, and meta-test channels only for the final scoring.

In [3]:
split = {"meta_train": sc.META_TRAIN, "meta_val": sc.META_VAL, "meta_test": sc.META_TEST,
         "eval_channels": sc.EVAL_CHANNELS, "sensitivity_channels": sc.SENSITIVITY_CHANNELS,
         "excluded": sc.EXCLUDED}
tr, va, te = map(set, (sc.META_TRAIN, sc.META_VAL, sc.META_TEST))
assert not (tr & va or tr & te or va & te), "split overlaps"
assert tr | va | te == set(data), "split does not cover exactly the 54 SMAP channels"
print({k: len(v) for k, v in split.items() if isinstance(v, list)})
for ch, why in sc.EXCLUDED.items():
    print(f"excluded {ch}: {why}")

{'meta_train': 39, 'meta_val': 6, 'meta_test': 9, 'eval_channels': 7, 'sensitivity_channels': 1}
excluded D-12: only 312 training timesteps (29 normal windows at stride 10): too few to draw support sets and hold out normal data. Label-free reason.
excluded P-4: sensor-dropout (flat-line) anomaly that a reconstruction model cannot flag. This reason was found by inspecting the test anomalies in the original notebook 04, so the exclusion is post hoc. P-4 is still scored and reported separately as a sensitivity row.


## 3 — How much clipping happens

Scaling uses the training file's range. Test values outside that range are clipped to
[0, 1], as in the original. Clipping can shrink large anomalies, so we report how often it
happens on the evaluation channels.

In [4]:
csv, TRAIN_DIR, TEST_DIR = sc.find_smap_raw()
rows = []
for ch in sc.EVAL_CHANNELS + sc.SENSITIVITY_CHANNELS:
    d = data[ch]
    te_raw = np.load(os.path.join(TEST_DIR, f"{ch}.npy"))
    rng = np.where(d["scaler_max"] > d["scaler_min"], d["scaler_max"] - d["scaler_min"], 1.0)
    unclipped = (te_raw - d["scaler_min"]) / rng
    clipped = (unclipped < 0) | (unclipped > 1)
    y = d["labels"].astype(bool)
    rows.append({"channel": ch, "clipped_values_all": clipped.mean(),
                 "clipped_values_in_anomalies": clipped[y].mean() if y.any() else np.nan,
                 "clipped_values_in_normal": clipped[~y].mean()})
print(pd.DataFrame(rows).set_index("channel").round(4).to_string())

         clipped_values_all  clipped_values_in_anomalies  clipped_values_in_normal
channel                                                                           
E-3                  0.0001                       0.0000                    0.0001
D-7                  0.0066                       0.0188                    0.0000
E-6                  0.0001                       0.0073                    0.0000
D-6                  0.0000                       0.0000                    0.0000
T-2                  0.0000                       0.0000                    0.0000
A-6                  0.0000                       0.0000                    0.0000
D-3                  0.0168                       0.0400                    0.0026
P-4                  0.0001                       0.0014                    0.0000


## 4 — What each evaluation channel contains

In [5]:
rows = []
for ch in sc.EVAL_CHANNELS + sc.SENSITIVITY_CHANNELS:
    d = data[ch]
    rows.append({"channel": ch, "train_len": len(d["train"]), "test_len": len(d["test"]),
                 "normal_windows": len(d["normal_windows"]),
                 "legacy_anomaly_windows": len(d["legacy_anomaly_windows"]),
                 "test_anomaly_fraction": d["labels"].mean(),
                 "ranges": labels.loc[ch, "sequences"]})
print(pd.DataFrame(rows).set_index("channel").round(3).to_string())

         train_len  test_len  normal_windows  legacy_anomaly_windows  test_anomaly_fraction                                     ranges
channel                                                                                                                               
E-3           2880      8307             286                     637                  0.387                             [[5094, 8306]]
D-7           2583      7642             256                     535                  0.354                             [[4940, 7641]]
E-6           2880      8300             286                      14                  0.008                             [[5610, 5675]]
D-6           2594      7884             257                      17                  0.010                             [[4870, 4950]]
T-2           2855      8625             283                     352                  0.207                             [[6840, 8624]]
A-6            682      4453              66           

## 5 — Save

In [6]:
cfg = {"window": sc.WINDOW, "stride_normal": sc.STRIDE_NORMAL, "stride_anomaly_legacy": sc.STRIDE_ANOMALY,
       "scaling": "MinMax per channel, fitted on the training file only; clipped to [0, 1]",
       "labels": "per timestep, anomaly ranges inclusive at both ends; P-2 CSV rows merged",
       "channels": len(data)}
with open(os.path.join(OUT, "smap_prepared.pkl"), "wb") as f:
    pickle.dump({"data": data, "sequences": labels["sequences"].to_dict(), "config": cfg}, f)
sc.save_json(os.path.join(OUT, "smap_split.json"), split)
sc.save_json(os.path.join(OUT, "smap_preprocessing_config.json"), cfg)
print(sorted(f for f in os.listdir(OUT) if f.startswith("smap_")))

['smap_prepared.pkl', 'smap_preprocessing_config.json', 'smap_split.json']
